# dbt 5: Replay event-time microbatches

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dbt/dbt-05-microbatch.ipynb)

Run bounded event-time batches and replay a correction with real dbt and DuckDB.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q dbt-core~=1.9 dbt-duckdb~=1.9 duckdb~=2.0

## Setup and sample data

Create an isolated DuckDB project for a three-event delivery.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
from dbt.cli.main import dbtRunner

workspace = TemporaryDirectory()
project = Path(workspace.name)
models = project / "models"
models.mkdir()
warehouse = project / "warehouse.duckdb"
(project / "dbt_project.yml").write_text("name: handbook\nversion: '1.0'\nprofile: handbook\n")
(project / "profiles.yml").write_text(
    f"handbook:\n  outputs:\n    dev:\n      type: duckdb\n      path: '{warehouse}'\n      threads: 1\n"
    "  target: dev\n"
)


def dbt(*args, expect_success=True):
    result = dbtRunner().invoke([*args, "--project-dir", str(project), "--profiles-dir", str(project)])
    if result.success != expect_success:
        raise RuntimeError(f"Unexpected dbt result: {result.exception or result.result}")
    print("dbt completed:", result.success)


def query(sql):
    with duckdb.connect(str(warehouse)) as connection:
        print(connection.execute(sql).fetchall())

## Process one event-time interval at a time

An event-time microbatch processes a half-open interval such as February 1 through, but excluding, February 2. This lab runs real dbt builds. Python supplies the interval explicitly and DuckDB applies a keyed merge. This is portable orchestration of microbatches. Native dbt microbatch delegates interval planning and writes to a supporting adapter.

**Where it matters**

Rebuild a limited interval when a correction arrives or when a daily job misses a run.

**Watch out for**

A merge updates rows it sees. An empty interval or a source deletion needs an explicit replacement or deletion policy. Validate source-key uniqueness before merging.

In [ ]:
with duckdb.connect(str(warehouse)) as connection:
    connection.execute("create table raw_events(id int, event_day date, cents int)")
    connection.execute("insert into raw_events values (1, '2026-02-01', 1200), (2, '2026-02-02', 800), (3, '2026-02-03', 900)")
(models / "events.sql").write_text("""
{{ config(materialized='incremental', unique_key='id', incremental_strategy='merge') }}
select * from raw_events
where event_day >= date '{{ var("batch_start") }}'
  and event_day < date '{{ var("batch_end") }}'
""")

import json


def run_batch(start, end):
    dbt("run", "--vars", json.dumps({"batch_start": start, "batch_end": end}))


run_batch("2026-02-01", "2026-02-02")
run_batch("2026-02-02", "2026-02-03")
query("select id, cents from events order by id")

**What to notice:** Orders 1 and 2 are loaded. Order 3 belongs to the next interval.

## Replay a corrected interval

Replay the second date using the same boundary and key. The existing row for order 2 changes in place.

**Where it matters**

Correct a prior report without rebuilding unrelated days.

**Watch out for**

Freeze the same input to test idempotency. A replay after a source correction should produce intentionally changed output.

In [ ]:
with duckdb.connect(str(warehouse)) as connection:
    connection.execute("update raw_events set cents = 850 where id = 2")
for _ in range(2):
    run_batch("2026-02-02", "2026-02-03")
query("select id, cents from events order by id")

**What to notice:** The replay leaves two rows, with order 2 corrected to 850 and order 1 still at 1200.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Would an event at the exclusive end be included?

In [ ]:
# Try your solution here.

In [ ]:
print("It belongs to the next interval.")

## Key takeaways

- Rebuild a limited interval when a correction arrives or when a daily job misses a run.
- Correct a prior report without rebuilding unrelated days.

**Next:** [Semantic layer](../concepts/semantic-layer.ipynb)